# Bayesian phase difference estimation

Regenerate the original hydrogen models, run the phase-sensitive controlled evolutions, and fit the Bayesian update to actual finite observations.
The readout quantity used in the fit is Z. Each binding also retains the probability of observing zero.

Use the optional scientific packages installed in the Notebook's local environment.

In [ ]:
from importlib.metadata import version

Use the native molecular driver, state reference, and seeded local counts.

In [ ]:
from pathlib import Path

from IPython.display import Image, display

from examples.synthesis.community.paper_implementation_project.bpde.overview import (
    model,
    plots,
    quantum,
    workflow,
)
from examples.synthesis.community.paper_implementation_project.bpde.overview.main import (
    report,
)

print('Local SciPy:', version('scipy'))
figures = Path('notebook-figures')

Generate the one-qubit Hamiltonian at the source's 0.74 Å separation.

In [ ]:
result = workflow.run()
print(result.standard)

Display the fresh ground-state energy at all 30 distances.

In [ ]:
assert len(result.curve) == 30
plots.save(result, figures)
display(Image(filename=str(figures / 'energy-curve.png')))

Use the original adjacent-distance finite differences.

In [ ]:
assert len(model.differences(result.curve)[0]) == 29
display(Image(filename=str(figures / 'finite-difference.png')))

Keep the two displaced hydrogen branches, all Strang steps, and symbolic detuning.

In [ ]:
experiment = quantum.build((result.a, result.b), 1.0)
assert tuple(experiment.template.parameter_names) == ('delta_phase',)

Check the two-step short evolution and its actual packed resources.

In [ ]:
assert result.check.repetitions == 2
print(experiment.resources)

Read the three source offsets with one million trials each.

In [ ]:
assert result.check.energies == (0.1, 0.2, 0.3)
print('Short readout Z and exact zero probability:', result.check.z_estimates, result.check.exact_zero)

Use the same observable at every time and preserve the state reference.

In [ ]:
assert max(row.maximum_error for row in (result.check, result.diagnostic, *result.study)) < 1e-10

Evaluate the five-point detuning grid at time 200 and display the observed signal.

In [ ]:
assert result.diagnostic.repetitions == 400
display(Image(filename=str(figures / 'scan.png')))

Compare the best observed grid point with the finite-difference curve.

In [ ]:
display(Image(filename=str(figures / 'grid-maximum.png')))

Display all 20, 100 and 200 time cases on the same Z scale.

In [ ]:
assert tuple(row.repetitions for row in result.study) == (40, 200, 400)
display(Image(filename=str(figures / 'time-study.png')))

Estimate the Gaussian signal and update the Bayesian posterior from fresh finite draws.

In [ ]:
assert len(result.history) <= 30
report(result)

Compare the final gradient and uncertainty against the fresh FCI reference.

In [ ]:
display(Image(filename=str(figures / 'posterior.png')))